# Google Colab notebook for EMIT-L2A product download

Based on EMIT-L2B $CH_4$ product, this notebook retrieves the corresponding L2A acquisitions and downloads the hyperspectral data cube to pair it with the plume annotation.
This notebook uses Google Earth Engine and was ran on the Google Colab environment.

## Authenticate to the GEE project and connect to Drive

In [1]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import re
from glob import glob
import rasterio as rio
from google.colab import drive
from shapely.geometry import box

ee.Authenticate()
ee.Initialize(project='lithe-timer-424608-c4')

drive.mount("/content/drive/")

Mounted at /content/drive/


### Configs setup

In [2]:
%cd '/content/drive/MyDrive/EMIT_CH4/'

# Path to both the CSV and GPKG file
CSV_FILE = './emit_ch4_plumes_metadata.csv'
GPKG_FILE = './emit_ch4_plumes_metadata.gpkg'

df = pd.read_csv(CSV_FILE)
gdf = gpd.read_file(GPKG_FILE)

/content/drive/MyDrive/EMIT_CH4


## PoC download of one plume

In [3]:
EXP_ID = 88
exp_plume = gdf.iloc[EXP_ID]

single_plume_gdf = gpd.GeoDataFrame([exp_plume], geometry='geometry', crs="EPSG:4326")
utm_crs = single_plume_gdf.estimate_utm_crs()
plume_utm = single_plume_gdf.to_crs(utm_crs)

In [4]:
centroid_utm = plume_utm.geometry.centroid.iloc[0]
cx_m, cy_m = centroid_utm.x, centroid_utm.y
radius_m = 15360
minx, miny = cx_m - radius_m, cy_m - radius_m
maxx, maxy = cx_m + radius_m, cy_m + radius_m
bbox_utm = gpd.GeoSeries([box(minx, miny, maxx, maxy)], crs=utm_crs)
bbox_latlon = bbox_utm.to_crs("EPSG:4326").iloc[0]
ee_bbox = ee.Geometry.Polygon(list(bbox_latlon.exterior.coords))
roi_point = ee.Geometry.Point([exp_plume.geometry.centroid.x, exp_plume.geometry.centroid.y])

In [5]:
center_date = ee.Date.parse('YYYY-MM-dd\'T\'HH:mm:ss', exp_plume.startDate[:-1])
start_time = center_date.advance(-5, 'minute')
end_time = center_date.advance(5, 'minute')

In [6]:
idx = [i for i in range(286)]

In [7]:
collection = ee.ImageCollection("NASA/EMIT/L2A/RFL") \
        .filterDate(start_time, end_time) \
        .filterBounds(ee_bbox) \
        .select(idx)

In [8]:
collection.size().getInfo()

1

In [9]:
ee_crs = f"EPSG:{utm_crs.to_epsg()}"

In [10]:
image_mosaic = collection.mosaic().unmask(0)

In [11]:
roi_geometry = ee.Geometry(exp_plume.geometry.__geo_interface__)
mask_mosaic = ee.Image.constant(0).byte() \
    .paint(roi_geometry, 1) \
    .unmask(0)

In [15]:
image_512 = image_mosaic.reproject(crs=ee_crs, scale=60).clip(ee_bbox)
mask_512 = mask_mosaic.reproject(crs=ee_crs, scale=60).clip(ee_bbox)

In [ ]:
image_vis = {
    'bands': ['reflectance_36', 'reflectance_23', 'reflectance_12'],
    'min' : 0,
    'max' : 0.3,
}

mask_vis = {
    'min': 0,
    'max': 1,
    'palette': ['black', 'red']
}

map = geemap.Map(center=[exp_plume.geometry.centroid.y, exp_plume.geometry.centroid.x], zoom=11)
map.addLayer(image_512, image_vis, 'EMIT 512x512')
map.addLayer(mask_512, mask_vis, 'Plume Mask', opacity=0.5)

display(map)

## Expanding PoC to the full dataset

### Create the BBOXes used to crop the EMIT satellite image

In [ ]:
centroids = gdf.geometry.centroid
lon = centroids.x
lat = centroids.y
utm_zone = np.floor((lon + 180) / 6) + 1
epsg_base = np.where(lat >= 0, 32600, 32700)
gdf['utm_epsg'] = "EPSG:" + (epsg_base + utm_zone).astype(int).astype(str)
gee_bboxes = gpd.GeoSeries(index=gdf.index, crs="EPSG:4326")

for epsg, group in gdf.groupby('utm_epsg'):
  group_utm = group.to_crs(epsg)
  is_small = (group['width_km'] < 25) & (group['height_km'] < 25)
  small_utm_bboxes = group_utm[is_small].geometry.centroid.buffer(15360, cap_style=3) # 512 px at 60 m/px
  large_utm_bboxes = group_utm[~is_small].geometry.envelope.buffer(500, cap_style=3) # 500 m buffer for larger plumes
  combined_utm_bboxes = pd.concat([small_utm_bboxes, large_utm_bboxes])
  latlon_squares = gpd.GeoSeries(combined_utm_bboxes, crs=epsg).to_crs("EPSG:4326")
  gee_bboxes.loc[group.index] = latlon_squares

# For bigger, irregular plumes, the bounding box is a buffer of 500 m around the plume. For smaller plumes, the bounding box is a square of 512x512 px around the plume.
gdf['gee_bbox'] = gee_bboxes 
gdf['is_fixed_512'] = (gdf['width_km'] < 25) & (gdf['height_km'] < 25)

/tmp/ipykernel_525/956450794.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


## Download script

In [ ]:
from pathlib import Path

# create directories for images and masks on your GDrive.
IMG_DIR = Path('./EMIT_L2A_chips')
MASK_DIR = Path('./CH4_plumes_masks')

In [ ]:
# batch the GeoDataFrame into smaller chunks for processing
gdf_batch1 = gdf.iloc[:500]
gdf_batch2 = gdf.iloc[500:1000]
gdf_batch3 = gdf.iloc[1000:]

### Task sumbission loop

In [ ]:
bands = [i for i in range(285)]
for idx, plume in gdf_batch1.iterrows(): # change to gdf_batch2 or gdf_batch3 for other batches
  plume_id = plume['name']
  date_str = plume['name'][:15]

  img_out = IMG_DIR / f"{plume_id}.tif"
  mask_out = MASK_DIR / f"{plume_id}.tif"

  coords = list(plume.gee_bbox.exterior.coords) # get the coordinates of the bbox
  ee_bbox = ee.Geometry.Polygon(coords)

  ee_polygon = ee.Geometry(plume.geometry.__geo_interface__)

  center_date = ee.Date.parse('YYYYMMdd\'T\'HHmmss', date_str)

  collection = ee.ImageCollection("NASA/EMIT/L2A/RFL") \
        .filterDate(center_date.advance(-5, 'minute'), center_date.advance(5, 'minute')) \
        .filterBounds(ee_bbox) \
        .select(bands)

  image = collection.mosaic() \
            .unmask(0) \
            .reproject(crs=plume['utm_epsg'], scale=60) \
            .clip(ee_bbox) # get the HSI cube for the plume, reprojected to UTM and clipped to the bbox

  mask = ee.Image.constant(0).byte() \
            .paint(ee_polygon, 1) \
            .unmask(0) \
            .reproject(crs=plume['utm_epsg'], scale=60) \
            .clip(ee_bbox) # based on the plume geometry, create a binary mask, reprojected to UTM and clipped to the bbox

  try:
    task_img = ee.batch.Export.image.toDrive(
            image=image,
            description=f"IMG_{plume_id}", # Task name in GEE
            folder="EMIT_L2A_chips",       # Folder created in your Google Drive
            fileNamePrefix=plume_id,       # Actual file name
            region=ee_bbox,
            scale=60,
            crs=plume['utm_epsg'],
            maxPixels=1e10 # Prevents arbitrary pixel limit errors
        )
    task_img.start() # Submits to Google's servers and moves on 

    task_mask = ee.batch.Export.image.toDrive(
            image=mask,
            description=f"MASK_{plume_id}",
            folder="CH4_plumes_masks",
            fileNamePrefix=plume_id,
            region=ee_bbox,
            scale=60,
            crs=plume['utm_epsg'],
            maxPixels=1e10
        )
    task_mask.start()

    print(f"Submitted {plume_id} to GEE Batch queue.")

  except Exception as e:
      print(f"Failed to submit {plume_id}: {e}")

In [ ]:
import time
from collections import Counter
from IPython.display import clear_output

# Monitor the status of the tasks in the GEE Batch queue
try:
    while True:
        tasks = ee.batch.Task.list()[:1148]
        status_counts = Counter([task.state for task in tasks])

        clear_output(wait=True)

        print("LIVE GEE Task Monitor")
        print(f"Last updated: {time.strftime('%H:%M:%S')}\n")

        for state, count in status_counts.items():
            print(f"{state:10}: {count} tasks")

        if status_counts.get('READY', 0) == 0 and status_counts.get('RUNNING', 0) == 0:
            print("\nAll tasks have finished processing!")
            break

        time.sleep(60)

except KeyboardInterrupt:
    print("\nLive monitor stopped by user.")